In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv('../data/TCGA_Reports.csv')

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print("\nFirst patient ID:", df['patient_filename'].iloc[0])

Shape: (9523, 2)
Columns: ['patient_filename', 'text']

First patient ID: TCGA-BP-5195.25c0b433-5557-4165-922e-2c1eac9c26f0


In [3]:
brca_codes = {'A1','A2','A7','A8','AC','AN','AO','AQ','AR','B6','BH','C8','D8',
              'E2','E9','EW','GI','GM','HN','JL','LD','LL','LQ','MS','OK','OL',
              'PE','PL','S3','TV','UL','UU','V7','W8','WT','XX','Z7','3C','3J','4H','5L','5T'}

luad_codes = {'05','17','35','38','44','49','50','53','55','62','64','67','69',
              '71','73','75','78','80','83','86','91','93','95','97','99',
              '4B','J2','L4','L9','ME','MN','MP','NB','NJ','O1','S2','T6'}

prad_codes = {'2A','4L','4S','CH','EJ','FC','G9','H9','HC','HI','J4','J9',
              'KC','KK','M7','MG','SU','TK','TP','V1','V2','VN','VP',
              'WW','X4','XA','XJ','XK','XQ','Y6','YJ','YL','ZG','QU','UR'}

def get_cancer_type(patient_id):
    tss = patient_id.split('-')[1]
    if tss in brca_codes: return 'BRCA'
    if tss in luad_codes: return 'LUAD'
    if tss in prad_codes: return 'PRAD'
    return None

df['cancer_type'] = df['patient_filename'].apply(get_cancer_type)

print(df['cancer_type'].value_counts())
print("\nUnlabeled:", df['cancer_type'].isna().sum())

cancer_type
BRCA    1034
LUAD     488
PRAD     446
Name: count, dtype: int64

Unlabeled: 7555


In [4]:
df_labeled = df[df['cancer_type'].notna()].copy()
df_labeled = df_labeled.reset_index(drop=True)

print("Total labeled reports:", len(df_labeled))
print("\nCancer type distribution:")
print(df_labeled['cancer_type'].value_counts())

Total labeled reports: 1968

Cancer type distribution:
cancer_type
BRCA    1034
LUAD     488
PRAD     446
Name: count, dtype: int64


In [5]:
from sklearn.model_selection import train_test_split

df_train_val, df_test = train_test_split(
    df_labeled,
    test_size=0.10,
    random_state=42,
    stratify=df_labeled['cancer_type']
)

df_train, df_val = train_test_split(
    df_train_val,
    test_size=0.111, 
    random_state=42,
    stratify=df_train_val['cancer_type']
)

print("Train:", len(df_train))
print("Val:  ", len(df_val))
print("Test: ", len(df_test))

Train: 1574
Val:   197
Test:  197


In [6]:
print("Train distribution:")
print(df_train['cancer_type'].value_counts(normalize=True).round(3))

print("\nVal distribution:")
print(df_val['cancer_type'].value_counts(normalize=True).round(3))

print("\nTest distribution:")
print(df_test['cancer_type'].value_counts(normalize=True).round(3))

Train distribution:
cancer_type
BRCA    0.526
LUAD    0.248
PRAD    0.226
Name: proportion, dtype: float64

Val distribution:
cancer_type
BRCA    0.523
LUAD    0.249
PRAD    0.228
Name: proportion, dtype: float64

Test distribution:
cancer_type
BRCA    0.523
LUAD    0.249
PRAD    0.228
Name: proportion, dtype: float64


In [7]:
df_train.to_csv('../data/train.csv', index=False)
df_val.to_csv('../data/val.csv', index=False)
df_test.to_csv('../data/test.csv', index=False)

print("Saved train.csv, val.csv, test.csv to data/")

Saved train.csv, val.csv, test.csv to data/
